In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import pickle
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# El nombre de la carpeta
dataset_path = '/content/drive/MyDrive/RootKit/PlantVillage'

# Verificar que lo encuentra
if os.path.exists(dataset_path):
    print("Dataset encontrado")
else:
    print("No encontrado")

Dataset encontrado


In [ ]:
# 1. Función Mejorada: Extraer Histogramas de Color (16 features en lugar de 3)
def procesar_imagen_mejorada(ruta_imagen):
    img = cv2.imread(ruta_imagen)
    if img is None:
        return None

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # -- 1. Calcular la Severidad (Variable 'y') --
    if 'Tomato' in ruta_imagen or 'jitomate' in ruta_imagen.lower():
        lower_bound = np.array([10, 40, 40])
        upper_bound = np.array([35, 255, 255])
    elif 'Potato' in ruta_imagen or 'papa' in ruta_imagen.lower():
        lower_bound = np.array([0, 30, 20])
        upper_bound = np.array([25, 255, 200])
    elif 'Pepper' in ruta_imagen or 'pimiento' in ruta_imagen.lower():
        lower_bound = np.array([15, 50, 50])
        upper_bound = np.array([40, 255, 255])
    else:
        lower_bound = np.array([10, 50, 50])
        upper_bound = np.array([35, 255, 255])

    mask = cv2.inRange(hsv, lower_bound, upper_bound)
    severidad = np.count_nonzero(mask) / mask.size

    # -- 2. Calcular Features (Variable 'X'): Histograma de Tono (Hue) --
    # Dividimos los colores en 16 grupos (bins) para que el modelo tenga más datos
    hist_hue = cv2.calcHist([hsv], [0], None, [16], [0, 180])
    cv2.normalize(hist_hue, hist_hue) # Normalizar para que no afecte el tamaño de la foto

    # Convertir el histograma a una lista plana
    features = hist_hue.flatten().tolist()
    features.append(severidad) # Añadimos la severidad al final

    return features

In [ ]:
# 2. Recorrer el dataset PlantVillage
# IMPORTANTE: Cambia esta ruta a la carpeta donde tienes tus imágenes en Google Drive
ruta_dataset = dataset_path

datos_extraidos = []
print("Iniciando extracción de features... Esto puede tardar unos minutos.")

# Recorrer carpetas y archivos (Limitamos a unas cuantas para probar rápido, luego quita el límite)
contador = 0
for raiz, carpetas, archivos in os.walk(ruta_dataset):
    for archivo in archivos:
        if archivo.endswith(('.JPG', '.jpg', '.png')):
            ruta_completa = os.path.join(raiz, archivo)
            resultado = procesar_imagen_mejorada(ruta_completa)

            if resultado is not None:
                datos_extraidos.append(resultado)
                contador += 1
                if contador % 500 == 0:
                    print(f"{contador} imágenes procesadas...")

print(f"Extracción completada. Total de imágenes: {len(datos_extraidos)}")



Iniciando extracción de features... Esto puede tardar unos minutos.
500 imágenes procesadas...
1000 imágenes procesadas...
1500 imágenes procesadas...
2000 imágenes procesadas...
2500 imágenes procesadas...
3000 imágenes procesadas...
3500 imágenes procesadas...
4000 imágenes procesadas...
4500 imágenes procesadas...
5000 imágenes procesadas...
5500 imágenes procesadas...
6000 imágenes procesadas...
6500 imágenes procesadas...
7000 imágenes procesadas...
7500 imágenes procesadas...
8000 imágenes procesadas...
8500 imágenes procesadas...
9000 imágenes procesadas...
9500 imágenes procesadas...
10000 imágenes procesadas...
10500 imágenes procesadas...
11000 imágenes procesadas...
11500 imágenes procesadas...
12000 imágenes procesadas...
12500 imágenes procesadas...
13000 imágenes procesadas...
13500 imágenes procesadas...
14000 imágenes procesadas...
14500 imágenes procesadas...
15000 imágenes procesadas...
15500 imágenes procesadas...
16000 imágenes procesadas...
16500 imágenes procesada

In [ ]:
# 3. Construir el DataFrame dinámico
# Creamos nombres para las 16 columnas del histograma
columnas = [f'Hue_Bin_{i}' for i in range(16)] + ['Severidad']
df = pd.DataFrame(datos_extraidos, columns=columnas)

In [ ]:
# 4. Separar X e y
X = df.drop(columns=['Severidad'])
y = df['Severidad']

In [ ]:
# 5. Dividir datos, entrenar y evaluar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

print("\nEntrenando RandomForestRegressor...")
modelo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

predicciones = modelo.predict(X_test)
r2 = r2_score(y_test, predicciones)
mae = mean_absolute_error(y_test, predicciones)

print(f"\n--- Resultados Finales ---")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")


Entrenando RandomForestRegressor...

--- Resultados Finales ---
R2 Score: 0.8516
MAE: 0.0164


In [ ]:


# 6. Exportar si cumple la meta
if r2 > 0.70:
    ruta_exportacion = '/content/drive/MyDrive/RootKit/models/regresion_severidad.pkl'
    with open(ruta_exportacion, 'wb') as f:
        pickle.dump(modelo, f)
    print(f"\n¡Éxito! Modelo exportado como '{ruta_exportacion}'.")
else:
    print("\nEl R2 es menor a 0.70.")


¡Éxito! Modelo exportado como '/content/drive/MyDrive/RootKit/models/regresion_severidad.pkl'.
